# Output 3-1 — External stress sensitivity tables

Excel analogue: **Output 3-1 Stress-external** — one block per indicator,
rows = Baseline + A / B / C scenarios, columns = projection years.
(**Output 2-1** charts use the same series.)

Runs **A1** (historical averages) in Python, standard **B1 / B3 / B4 / B5 / B6**
from Input 6 + Input 7 ResFin, and loads **A2 / C1 / C3 / C4** ratios from the
workbook stress sheets until tailored runners land. **C2** is omitted when Input
6 marks natural disasters inapplicable (`n.a.` in Excel).

See `docs/08-stress-dsa.qmd`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.dsa import load_core
from lic_dsf.pv import load_input7_residual_params
from lic_dsf.stress import (
    load_cached_external_stress,
    load_input6_standard,
    run_a1_historical_external,
    run_standard_external_stress,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK

PosixPath('/home/sravan/excel-grapher/py-lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

In [2]:
macro, external, ext_base, _pub_base = load_core(WORKBOOK)
input6 = load_input6_standard(WORKBOOK)
residual = load_input7_residual_params(WORKBOOK)
external_stress = run_standard_external_stress(macro, external, input6, residual)
historical = run_a1_historical_external(macro, external, residual)
cached_stress = load_cached_external_stress(WORKBOOK)

first_proj = macro.inputs.first_projection_year
years = list(range(first_proj, first_proj + 11))
list(external_stress), list(cached_stress), years[0], years[-1]

(['B1_GDP', 'B3_Exports', 'B4_OtherFlows', 'B5_FX', 'B6_Combo'], 2024, 2034)

## Sensitivity tables (Output 3-1 shape)

Each block: rows = Baseline, A1/A2, B-tests, and tailored C-tests (when
applicable), columns = years 1–11 of the rating window.

In [3]:
B_SCENARIO_LABELS = {
    "B1_GDP": "B1. Real GDP growth",
    "B3_Exports": "B3. Exports",
    "B4_OtherFlows": "B4. Other flows",
    "B5_FX": "B5. Depreciation",
    "B6_Combo": "B6. Combination of B1-B5",
}

CACHED_SCENARIO_LABELS = {
    "A2_Custom": "A2. Alternative Scenario",
    "C1_CombinedCL": "C1. Combined contingent liabilities",
    "C3_Commodity": "C3. Commodity price",
    "C4_Market": "C4. Market Financing",
}


def _ratio_row(label: str, series: pd.Series, yrs: list[int]) -> pd.Series:
    return series.reindex(yrs).rename(label)


def stress_table_for_indicator(getter_name: str) -> pd.DataFrame:
    rows = [_ratio_row("Baseline", getattr(ext_base, getter_name)(), years)]
    rows.append(
        _ratio_row(
            "A1. Key variables at their historical averages",
            getattr(historical, getter_name)(),
            years,
        )
    )
    for key, label in CACHED_SCENARIO_LABELS.items():
        book = cached_stress.get(key)
        if book is not None:
            rows.append(_ratio_row(label, getattr(book, getter_name)(), years))
    for sid, label in B_SCENARIO_LABELS.items():
        book = external_stress[sid]
        rows.append(_ratio_row(label, getattr(book, getter_name)(), years))
    return pd.DataFrame(rows)


out_3_1 = {
    "PV of debt-to-GDP": stress_table_for_indicator("pv_ppg_external_to_gdp"),
    "PV of debt-to-exports": stress_table_for_indicator("pv_ppg_external_to_exports"),
    "Debt service-to-exports": stress_table_for_indicator(
        "ppg_debt_service_to_exports"
    ),
    "Debt service-to-revenue": stress_table_for_indicator(
        "ppg_debt_service_to_revenue"
    ),
}
list(out_3_1)

['PV of debt-to-GDP',
 'PV of debt-to-exports',
 'Debt service-to-exports',
 'Debt service-to-revenue']

In [4]:
out_3_1["PV of debt-to-GDP"]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,44.8846,43.1514,41.2265,40.1941,38.6043,37.6952,35.4235,32.9897,31.9558,31.4151,31.2078
B1. Real GDP growth,44.8846,46.9546,46.6468,45.4787,43.6799,42.6512,40.0808,37.3271,36.1573,35.5455,35.3109
B3. Exports,44.8846,48.0971,55.8763,54.7741,53.0815,52.0285,48.9165,44.4894,41.6739,39.4934,37.7703
B4. Other flows,44.8846,46.7521,48.3838,47.3282,45.7052,44.7324,41.8912,38.4527,36.5060,35.1198,34.1271
B5. Depreciation,44.8846,51.7966,46.3538,45.0641,43.0763,41.9764,39.5571,37.3019,36.7452,36.7580,37.1438
B6. Combination of B1-B5,44.8846,51.1009,50.8940,49.7077,47.8823,46.8131,43.6635,40.3468,38.6712,37.5931,36.9267


In [5]:
out_3_1["PV of debt-to-exports"]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,109.0647,103.4689,99.5562,98.4192,95.2973,93.7093,89.4654,84.9560,83.8739,83.5280,83.6537
B1. Real GDP growth,109.0647,103.4689,99.5562,98.4192,95.2973,93.7093,89.4654,84.9560,83.8739,83.5280,83.6537
B3. Exports,109.0647,128.6924,166.9198,165.9134,162.0973,160.0026,152.8297,141.7294,135.3096,129.8989,125.2452
B4. Other flows,109.0647,112.1029,116.8400,115.8878,112.8262,111.2037,105.8003,99.0244,95.8165,93.3781,91.4791
B5. Depreciation,109.0647,97.9761,88.3041,87.0467,83.8855,82.3201,78.8121,75.7791,76.0818,77.0991,78.5440
B6. Combination of B1-B5,109.0647,117.9703,121.3466,120.1739,116.7047,114.9036,108.8809,102.5872,100.2151,98.6893,97.7310


In [6]:
out_3_1["Debt service-to-exports"]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,17.1681,16.5074,15.1705,16.2756,18.5608,15.8922,19.6644,19.4605,16.5719,15.0800,13.3194
B1. Real GDP growth,17.1681,16.5074,15.1705,16.2756,18.5608,15.8922,19.6644,19.4605,16.5719,15.0800,13.3194
B3. Exports,17.1681,18.5538,20.2809,23.7217,26.6217,23.2286,30.0993,33.7600,29.7740,27.5471,24.9755
B4. Other flows,17.1681,16.5074,15.8168,17.5864,19.8727,17.2001,22.2574,23.2894,20.2648,18.6320,16.7289
B5. Depreciation,17.1681,16.5074,14.7593,15.4222,17.7066,15.0407,17.9952,16.9663,14.1663,12.7662,11.0986
B6. Combination of B1-B5,17.1681,17.4709,17.6935,18.9930,21.5577,18.5612,24.1247,23.9245,20.6098,18.8595,16.8070


In [7]:
out_3_1["Debt service-to-revenue"]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,39.0015,37.8686,33.7022,35.3500,40.2087,34.3537,42.3044,41.5353,35.0544,31.7428,27.9829
B1. Real GDP growth,39.0015,37.8686,33.7022,35.3500,40.2087,34.3537,42.3044,41.5353,35.0544,31.7428,27.9829
B3. Exports,39.0015,37.8686,35.5365,40.6376,45.4871,39.6042,51.0727,56.8323,49.6749,45.7350,41.3856
B4. Other flows,39.0015,37.8686,35.1381,38.1972,43.0509,37.1809,47.8827,49.7077,42.8661,39.2197,35.1458
B5. Depreciation,39.0015,49.2291,42.6253,43.5454,49.8659,42.2670,50.3275,47.0754,38.9557,34.9341,30.3123
B6. Combination of B1-B5,39.0015,43.5489,40.2867,42.2802,47.8649,41.1230,53.1934,52.3357,44.6822,40.6880,36.1900
